# 🔒 GIADA Task 3e — conferma full-repair
Verifica e riuso degli esatti checkpoint Task 3d; nessun training e apertura sealed una sola volta.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_3e'); GIADA_REPO=WORK/'giada'; TEACHER_REPO=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip(); print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
import torch
assert torch.cuda.is_available(),'La Task 3e richiede una GPU CUDA Kaggle.'
from src.giada_teacher import ExtractedGateFormula,JointGateFullRepairConfirmationConfig,prepare_joint_gate_dataset,prepare_joint_gate_generalization_diagnosis,freeze_full_repair_from_task3d,evaluate_frozen_full_repair
from src.giada_teacher.joint_gate_full_repair_confirmation import EXPECTED_TASK3D_ARCHIVE_SHA256,EXPECTED_TASK3D_REPORT_SHA256
prereg=json.loads((GIADA_REPO/'experiments/teacher_joint_gate_full_repair_confirmation_preregistration_v1.json').read_text())
display({'gpu':torch.cuda.get_device_name(0),'preregistration':prereg})


In [ ]:
def file_sha(path):
    d=hashlib.sha256()
    with Path(path).open('rb') as h:
        for chunk in iter(lambda:h.read(1024*1024),b''): d.update(chunk)
    return d.hexdigest()
INPUT_ROOT=Path('/kaggle/input'); override=os.environ.get('GIADA_TASK3D_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
    candidates += list(INPUT_ROOT.rglob('giada_joint_m_h_generalization_diagnosis.zip'))
    candidates += list(INPUT_ROOT.rglob('archive.zip'))
    candidates += [p.parent for p in INPUT_ROOT.rglob('final_report.json') if (p.parent/'development_checkpoints.pt').is_file()]
def exact(path):
    try:
        if path.is_file(): return file_sha(path)==EXPECTED_TASK3D_ARCHIVE_SHA256
        return file_sha(path/'final_report.json')==EXPECTED_TASK3D_REPORT_SHA256 and (path/'development_checkpoints.pt').is_file()
    except Exception: return False
TASK3D_SOURCE=next((p.resolve() for p in candidates if p.exists() and exact(p)),None)
assert TASK3D_SOURCE is not None,'Artefatto esatto giada_joint_m_h_generalization_diagnosis non trovato negli Input Kaggle.'
print({'task3d_source':str(TASK3D_SOURCE)})


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_joint_m_h_full_repair_confirmation')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
base=prepare_joint_gate_dataset(formula); development_bundle=prepare_joint_gate_generalization_diagnosis(formula,base)
config=JointGateFullRepairConfirmationConfig()
display({'candidate':'full_repair shared width23 step50000','seeds':config.seeds,'sealed_not_materialized_yet':True})


## ✅ Verifica sorgente e freeze
Nessun dato sealed viene materializzato in questa fase.

In [ ]:
freeze_report=freeze_full_repair_from_task3d(development_bundle,OUTPUT_DIR,TASK3D_SOURCE,config,code_revision=REVISION)
display(freeze_report)
assert freeze_report['valid'] and freeze_report['source_verified'] and not freeze_report['retraining_performed'] and not freeze_report['sealed_accessed']


## 🧪 Apertura sealed una sola volta
Eseguire solo dopo il freeze valido.

In [ ]:
final=evaluate_frozen_full_repair(development_bundle,OUTPUT_DIR,config)
display({'valid':final['valid'],'sealed_contract':final['sealed_contract'],'decision':final['decision'],'selection_used_sealed':final['selection_used_sealed']})
assert final['valid'] and not final['selection_used_sealed']


## 📦 Download Blob/base64

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_joint_m_h_full_repair_confirmation','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
